<a href="https://colab.research.google.com/github/kader-xai/llm-from-scratch/blob/main/notebooks/todo/Chapter01_dataloader_todo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🧱 **LLM From Scratch · Chapter 01 — Dataloader — Text → Batches**

> ✍️ TODO notebook — fill in the `# YOUR CODE HERE` blanks, then check against the answer.

## Goal

A language model trains on `(context, next-token)` pairs. We need to:
1. Build a **character-level tokenizer** (map each unique char ↔ an integer).
2. Encode the whole corpus into one long tensor of ids.
3. Sample random **batches** of `(x, y)` where `y` is `x` shifted one position right.

Character-level keeps things simple — vocab is just the unique characters (~65 for Shakespeare).

In [ ]:
# ── Setup (run first) ─────────────────────────────────────────────────
# Pure PyTorch — no extra installs needed beyond torch (preinstalled on Colab).
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

In [ ]:
# Download the training corpus (Tiny Shakespeare, ~1 MB) if not present.
import os, urllib.request
if not os.path.exists("input.txt"):
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, "input.txt")
text = open("input.txt").read()
print(f"corpus: {len(text):,} characters")
print(text[:200])

## 1 · Character tokenizer

`stoi` maps string→int, `itos` maps int→string. `encode` and `decode` use them.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
# YOUR CODE HERE: build stoi (char->int) and itos (int->char) dicts
stoi = ...
itos = ...
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("vocab_size:", vocab_size)
print("encode('hi there'):", encode("hi there"))
print("decode back:", decode(encode("hi there")))

## 2 · Encode the corpus + train/val split

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
print("train tokens:", len(train_data), "| val tokens:", len(val_data))

## 3 · Batch sampler

For a context length `block_size`, pick `batch_size` random start positions. `x` is the context,
`y` is `x` shifted by one — the target the model must predict at each position.

In [ ]:
block_size = 8
batch_size = 4

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    # YOUR CODE HERE: x = contexts of length block_size; y = same shifted right by 1
    x = ...
    y = ...
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("x:", xb.shape, "y:", yb.shape)
print("first context:", decode(xb[0].tolist()))
print("first target :", decode(yb[0].tolist()))

✅ You can now turn text into training batches. Next: turn ids into vectors.